# Install packages

In [1]:
!pip install geopandas

DEPRECATION: Loading egg at c:\users\mchlc\appdata\local\programs\python\python312\lib\site-packages\dcnet-0.0.1-py3.12.egg is deprecated. pip 23.3 will enforce this behaviour change. A possible replacement is to use pip for package installation..


In [2]:
import geopandas as gpd
import pandas as pd

c:\Users\mchlc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\mchlc\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


# Data processing

### [COMMERICAL FISHERIES] This reduces fisheries data size by aggregating by polygons.

Input: `../public/fisheriesdata/20260216_tidied_comm_ev.gpkg` (commercial fisheries data from Alemarie - removed to save space.)

Output: `../public/fisheriesdata/FOR_PROJECT/09182026_comm_EV_byCatchmentArea.geojson` (geoJSON, ready to be used)

In [ ]:
import math
geojson_df = gpd.read_file("../public/fisheriesdata/20260216_tidied_comm_ev.gpkg")

crs = geojson_df.crs or "EPSG:4326"
def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    
    return list(group_data)

geojson_df.drop(["county", "exchange_value_formatted"], inplace=True, axis=1)
geojson_df["area_id"] = geojson_df["area_id"].apply(lambda x: "Catchment Area " + str(x))
geojson_df["exchange_value"] = geojson_df["exchange_value"].fillna(-1.0)
geojson_df["exchange_value"] = geojson_df["exchange_value"].apply(lambda x: math.trunc(x * 100) / 100)

x = geojson_df.groupby(["geometry"], as_index=False).agg(groupIntoArray)

# check if it's all good.

new_gdf = gpd.GeoDataFrame(x, geometry="geometry", crs=crs)
#new_gdf.to_file("collapsed_comm_ev.geojson", driver="geoJSON")

new_gdf.to_file("../public/fisheriesdata/USED_IN_PROJECT/10_14_2026_comm_EV_byCatchmentArea.geojson", driver="geoJSON", COORDINATE_PRECISION=5)
x.head(2)

,geometry,year,area_id,county_olelo,species_group,ecosystem_type,exchange_value
0,"MULTIPOLYGON (((-160.5084 21.66031, -160.51124...","[2021, 2021, 2021, 2021, 2021, 2021, 2021, 202...",Catchment Area 508,Kauaʻi,"[Deep 7 Bottomfish, Deep 7 Bottomfish, Deep 7 ...","[All Ecosystems, Inshore — Reef, Coastal — Ope...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 0.0, 0.0,..."
1,"MULTIPOLYGON (((-160.37768 21.71885, -160.3760...","[2021, 2021, 2021, 2021, 2021, 2021, 2021, 202...",Catchment Area 528,Kauaʻi,"[Deep 7 Bottomfish, Deep 7 Bottomfish, Deep 7 ...","[All Ecosystems, Inshore — Reef, Coastal — Ope...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 0.0, 0.0,..."


In [ ]:
def is_valid_df(x):
    must_have = {"geometry", "year", "area_id", "county_olelo", "species_group", "ecosystem_type", "exchange_value"}
    current_columns = set(x.columns)

    if (must_have.issubset(current_columns)):
        print("✅ All columns are valid!")
    else:
        print("❌ The following columns are missing: " + str(must_have - current_columns))
        print("We require the following columns: " + str(must_have))
        return False;

    are_rows_valid = True;
    for row in x.index:
        if ((len(x.loc[row, "year"]) == len(x.loc[row, "species_group"])) and (len(x.loc[row, "species_group"]) == len(x.loc[row, "ecosystem_type"])) and (len(x.loc[row, "ecosystem_type"]) == len(x.loc[row, "exchange_value"]))):
            pass
        else:
            print("❌ Row " + str(row) + " is invalid!")
            cols_to_compare = ["year", "species_group", "ecosystem_type", "exchange_value"]
            for col in cols_to_compare:
                print("Length of '" + col + "' is: " + str(len(x.loc[row, col])))
            print("Array lengths mismatch.")
            print("=====")
            are_rows_valid = False;

    if (are_rows_valid):
        print("✅ All rows are valid!")
    else:
        print("❌ Some columns are invalid. See above.")

    return are_rows_valid;

is_valid_df(x)

❌ The following columns are missing: {'hargga'}
We require the following columns: {'year', 'county_olelo', 'geometry', 'area_id', 'exchange_value', 'species_group', 'ecosystem_type', 'hargga'}


False

### [NONCOMMERCIAL FISHERIES] Apply the same collapsing.

Input: `../public/fisheriesdata/20260216_tidied_noncomm_ev.gpkg` (noncommercial fisheries data from Alemarie)

Output: `../public/fisheriesdata/USED_IN_PROJECT/10_14_2026_comm_EV_byIsland.geojson` (geoJSON, ready to be used)

In [ ]:
geojson_df = gpd.read_file("../public/fisheriesdata/20260216_tidied_noncomm_ev.gpkg")

crs = geojson_df.crs or "EPSG:4326"
def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    
    return list(group_data)

geojson_df.drop(["island", "county", "exchange_value_formatted"], inplace=True, axis=1)
geojson_df["exchange_value"] = geojson_df["exchange_value"].fillna(-1.0)
x = geojson_df.groupby(["geometry"], as_index=False).agg(groupIntoArray)

# check if it's all good.

new_gdf = gpd.GeoDataFrame(x, geometry="geometry", crs=crs)
#new_gdf.to_file("collapsed_comm_ev.geojson", driver="geoJSON")

new_gdf.to_file("../public/fisheriesdata/USED_IN_PROJECT/10_14_2026_comm_EV_byIsland.geojson", driver="geoJSON", COORDINATE_PRECISION=5)
x

### [TERRESTRIAL MOKU EXTENTS] Merge the moku extents data, also by aggregating by polygon.

Input: `../public/mokuextentsdata/mokus_extents.geojson` (from who?)

Output: `../public/mokuextentsdata/USED_IN_PROJECT/2026_10_07_terrestrial_extents_byMoku.geojson` (geoJSON, ready to be used)

In [ ]:
df = gpd.read_file("../public/mokuextentsdata/mokus_extents.geojson")

crs = df.crs or "EPSG:4326"
def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    if (len(set(group_data)) == 0):
        return []
    
    final_list = list(group_data)
    return [x if not x is None else 0 for x in final_list]

df.drop(["fid", "name2", "moku", "island"], inplace=True, axis=1)
x = df.groupby(["geometry"], as_index=False).agg(groupIntoArray)

#marine_df = x.loc[x["realm"] == "Marine"]
terrestrial_df = x.loc[x["realm"] == "Terrestrial"]

terrestrial_df["total_area_km2"] = terrestrial_df.apply(lambda x: sum(x["area_km2"]) / len(set(x["year"])) if len(x["area_km2"]) > 0 else 0, axis=1)

#new_marine = gpd.GeoDataFrame(marine_df, geometry="geometry", crs=crs)
new_terrestrial = gpd.GeoDataFrame(terrestrial_df, geometry="geometry", crs=crs)

new_terrestrial.to_file("../public/mokuextentsdata/USED_IN_PROJECT/2026_10_07_terrestrial_extents_byMoku.geojson", driver="geoJSON", COORDINATE_PRECISION=5)

### [MARINE MOKU EXTENTS] Merge the moku extents data, also by aggregating by polygon.

Input 1: `../public/mokuextentsdata/mokus_extents.geojson` (from who?)

Input 2: `../public/mokuextentsdata/EDS_Timeseries_moku_2024-07-02.csv` (from Louis Chua)

Output: `../public/mokuextentsdata/USED_IN_PROJECT/2026_10_23_marine_extents_byMoku.geojson` (geoJSON, ready to be used)

In [31]:
import math

data_df = pd.read_csv("../public/mokuextentsdata/EDS_Timeseries_moku_2024-07-02.csv")

# it looks like "NAME2" is the primary key.
keep_variables = ["NAME2",
                  "StartDate",
                  "mean_Chlorophyll_A_ESA_OC_CCI_v6.0_8Day_SPAN",
                  "mean_Chlorophyll_A_NPP_VIIRS_Weekly_SPAN",
                  "DHW.MeanMax_Degree_Heating_Weeks_CRW_Daily_SPAN",
                  "mean_Kd490_ESA_OC_CCI_8Day_SPAN",
                  "mean_PAR_NASA_VIIRS_8Day_SPAN",
                  "mean_Sea_Surface_Temperature_CRW_Daily_SPAN",
                  "mean_Wave_Height_WW3_HI_Hourly_SPAN"]

new_variable_names = {
    "NAME2": "name2",
    "StartDate": "year",
    "mean_Chlorophyll_A_ESA_OC_CCI_v6.0_8Day_SPAN": "mean_Chlorophyll_A_ESA",
    "mean_Chlorophyll_A_NPP_VIIRS_Weekly_SPAN": "mean_Chlorophyll_A_NPP",
    "DHW.MeanMax_Degree_Heating_Weeks_CRW_Daily_SPAN": "mean_max_degree_weeks",
    "mean_Kd490_ESA_OC_CCI_8Day_SPAN": "mean_Kd490_ESA",
    "mean_PAR_NASA_VIIRS_8Day_SPAN": "mean_PAR_NASA",
    "mean_Sea_Surface_Temperature_CRW_Daily_SPAN": "mean_sea_temp",
    "mean_Wave_Height_WW3_HI_Hourly_SPAN": "mean_wave_height",
}

def groupIntoArray(group_data):
    if (len(set(group_data)) == 1):
        return list(group_data)[0]
    if (len(set(group_data)) == 0):
        return []
    
    final_list = list(group_data)

    # trunc to 3 decimal places
    return [math.trunc(float(x) * 1000) / 1000 if not x is None else 0 for x in final_list]

# ===== start processing =====

new_data_df = data_df[keep_variables]
new_data_df.rename(columns=new_variable_names, inplace=True)
new_data_df["year"] = new_data_df["year"].apply(lambda x: x.split("-")[0])
# new_data_df["NAME2"] = new_data_df["NAME2"].apply(lambda x: x.split(" ")[0].capitalize())
new_data_df.head(2)

grouped_by_moku = new_data_df.groupby(["name2"], as_index=False).agg(groupIntoArray)
grouped_by_moku.head(2)

,name2,year,mean_Chlorophyll_A_ESA,mean_Chlorophyll_A_NPP,mean_max_degree_weeks,mean_Kd490_ESA,mean_PAR_NASA,mean_sea_temp,mean_wave_height
0,EWA,"[2013.0, 2016.0, 2019.0]","[0.226, 0.186, 0.152]","[0.114, 0.102, 0.11]","[0.0, 0.0, 1.078]","[0.046, 0.042, 0.037]","[43.06, 43.73, 45.813]","[25.666, 26.137, 26.34]","[1.053, 0.893, 0.924]"
1,HALELEA,"[2013.0, 2016.0, 2019.0]","[0.172, 0.18, 0.198]","[0.218, 0.186, 0.224]","[0.0, 0.0, 2.317]","[0.039, 0.04, 0.041]","[42.913, 42.048, 43.661]","[25.191, 25.567, 25.915]","[1.91, 2.162, 1.894]"


In [33]:
# Read the geojson file and process it.
geo_df = gpd.read_file("../public/mokuextentsdata/marineandmokuextents.geojson")
crs = geo_df.crs or "EPSG:4326"
geo_df = geo_df[["realm", "name2", "moku_olelo", "island_olelo", "geometry"]]
geo_df_marine = geo_df.loc[geo_df["realm"] == "Marine"]

# merge it with the CSV
merged_df = pd.merge(geo_df_marine, grouped_by_moku, on="name2", how="inner")
merged_df.drop(["realm", "name2"], inplace=True, axis=1)

new_marine=gpd.GeoDataFrame(merged_df, geometry="geometry", crs=crs)
new_marine.to_file("../public/mokuextentsdata/USED_IN_PROJECT/2026_10_23_marine_conditions_byMoku.geojson", driver="geoJSON", COORDINATE_PRECISION=5)